# Question 3 - Write Your First Test

This question introduces pytest testing for the SQL agent. We determine the expected count directly from DuckDB, so the test does not rely on the LLM for its ground truth.

```text
Known DuckDB answer
        ↓
run the SQL agent
        ↓
SQLResult
        ↓
pytest assertions
        ↓
PASS / FAIL
```

Run this notebook from `week3_homework/` with the project dependencies and `OPENAI_API_KEY` available in the environment. The async test lives in `test_agent.py`; the cells below explain its essential steps.

In [1]:
from sql_tools import con

verification_query = "SELECT COUNT(*) FROM trips WHERE passenger_count > 5;"
expected_count = con.execute(verification_query).fetchone()[0]
print(verification_query)
print("Expected DuckDB count:", expected_count)

SELECT COUNT(*) FROM trips WHERE passenger_count > 5;
Expected DuckDB count: 22413


In [2]:
from sql_agent import agent, SQLResult

result = await agent.run("How many trips had more than 5 passengers?")
output = result.output
assert isinstance(output, SQLResult)
print("sql_query:")
print(output.sql_query)
print("result_text:")
print(output.result_text)
print("Expected DuckDB count:", expected_count)

                 pydantic-ai v2.63.0 • Python 3.13.15

      / \        agent: agent • model: openai:gpt-4o-mini • output: SQLResult • tools: 2
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


sql_query:
SELECT COUNT(*) AS trips_with_more_than_5_passengers 
FROM trips 
WHERE passenger_count > 5;
result_text:
22413
Expected DuckDB count: 22413


In [3]:
assert isinstance(output.sql_query, str)
assert output.sql_query.strip()
assert str(expected_count) in output.result_text
print("PASS: SQL is non-empty and result_text contains the exact DuckDB count.")

PASS: SQL is non-empty and result_text contains the exact DuckDB count.


The pytest version uses the fixed count independently verified above as `EXPECTED_COUNT`. Run it from this folder with:

```bash
uv run pytest test_agent.py -s
```

This first test checks a non-empty SQL query and the expected count in the result. Later questions add other checks.

**Number of trips with more than 5 passengers: `22413`.**

The count comes directly from DuckDB. The live agent returned the same count, and all demonstrated assertions passed. The pytest test also passed (`1 passed`).